In [ ]:
import numpy as np
import os, sys
import matplotlib.pyplot as plt

from PyGEECSPlotter.scan_data_analyzers.scan_data_analysis import ScanDataAnalyzer
from PyGEECSPlotter.utils.navigation_utils import get_top_dir, generate_sfilename_list_from_scans_dir

import ipywidgets as widgets
import datetime

import pandas as pd

In [ ]:
# Can use search_replace to change column names to more useful names
# Can use column_math to quickly turn column values into more useful quantites (e.g. stage positions)

# column_math_filename = '/Users/alexpicksley/Documents/GitHub/bella_analysis/user_data/2024-02-29_columnmath_testguiding.txt'
# search_replace_filename = '/Users/alexpicksley/Documents/GitHub/bella_analysis/user_data/2024-02-29_searchreplace.txt'



In [ ]:
experiment_dir = r'Z:/data/Thomson/'
# experiment_dir = r'D:/Server/Data/PW/'
style = {}

start_date_widget = widgets.DatePicker(description='Start Date', value = datetime.date.today(), style=style, disabled=False)
end_date_widget = widgets.DatePicker(description='End Date', value = datetime.date.today(), style=style, disabled=False)
dates_box = widgets.HBox([start_date_widget, end_date_widget])
dates_box

In [ ]:
start_date = start_date_widget.value
end_date = end_date_widget.value

date_list = pd.date_range(start_date, end_date, freq='D')
year_month_day_list = [(d.year, d.month, d.day) for d in date_list]

year_month_day_list

In [ ]:

sfilename_list = []
for i in range(len(year_month_day_list)):
    year, month, day = year_month_day_list[i]
    top_dir = get_top_dir(experiment_dir, year, month, day)
    
    tmp_list = generate_sfilename_list_from_scans_dir(top_dir)
    for sfilename in tmp_list:
        if os.path.exists(sfilename):
            sfilename_list.append( sfilename )

In [ ]:
# Load the Scalar Data for that scan in, and filter any data if desired

all_parameters_list  = []
for sfilename in sfilename_list:
    scan_data = ScanDataAnalyzer(sfilename=sfilename)
    
    scan_data.load_scan_data(
        # search_replace_filename=search_replace_filename, 
        # column_math_filename=column_math_filename,
    )

    for col_name in scan_data.data.columns:
        all_parameters_list.append(col_name) if col_name not in all_parameters_list else all_parameters_list



In [ ]:
parameters_widget = widgets.SelectMultiple(options=all_parameters_list, rows=20, description='All Parameters', style=style, 
                                          layout=widgets.Layout(width='75%'), disabled=False)
parameters_widget

In [ ]:
parameter_list_from_widget = list(parameters_widget.value)
# parameter_list_manual = ['Cell Length (mm)', 'Compressor Position (um)', 'Gas Jet On ?', 'HTT-WFS-400nm himg exists', 
#                          'HTT-WFS-400nm png file exists', 'HTT-WFS-800nm himg exists', 'HTT-WFS-800nm png file exists', 
#                          'Lamp Timing (us)', 'Pressure Measured (mbar)', 'Pressure Measured (torr)', 'Pressure Set (mbar)', 
#                          'Pressure Set (torr)', 'Probe Delay (ns)']
# parameter_list_manual = [ 'Gas Jet On ?', 'Drive Pulse Delay (ns)', 'Gas Jet Backing Pressure (psi)','Compression', 
#                          'CAM-2BL-F3M5 mean counts T', 'HPD_Energy', 'HPDmodeMaxCounts',
#                          '1BL Delay', 'Amp2Attenuator', 'Amplifier 1 [mJ]', 'Amplifier 2 [J]', 
#                          'Amplifier 3 [J]', 'Comp-2BL-separation [mm]', 
#                           'HPD stage position','HPD2 stage position',  
#                          'Input Delta R (um)', 'Input Delta X (um)', 'Input Delta Y (um)', 'Jet Length (cm)', 
#                          ]

parameter_list_manual = []
parameter_list = parameter_list_from_widget + parameter_list_manual
print(parameter_list)

In [ ]:
columns = ['year', 'month', 'day', 'scan', '# shots', 'scan_parameter', 'scan_info_txt']
for parameter in parameter_list:
    columns.append('%s MEAN' %parameter)
#     columns.append('%s STD' %parameter)
columns.append('sfilename')

scan_sheet = pd.DataFrame(columns=columns)

In [ ]:
for i in range(len(sfilename_list)):
    sfilename = sfilename_list[i]
    scan_data = ScanDataAnalyzer(sfilename=sfilename)
    
    scan_data.load_scan_data(
        # search_replace_filename=search_replace_filename, 
        # column_math_filename=column_math_filename,
    )
    
    if not scan_data.aborted:
        scan_info_text = scan_data.get_scan_info_text()
    
        scan_sheet.loc[i, 'year'] = scan_data.year
        scan_sheet.loc[i, 'month'] = scan_data.month
        scan_sheet.loc[i, 'day'] = scan_data.day
        scan_sheet.loc[i, 'scan'] = scan_data.scan
        scan_sheet.loc[i, '# shots'] = len(scan_data.data)
        scan_sheet.loc[i, 'scan_parameter'] = scan_data.scan_parameter
        scan_sheet.loc[i, 'scan_info_txt'] = scan_info_text
        scan_sheet.loc[i, 'sfilename'] = sfilename
    
    for parameter in parameter_list:
        if parameter in scan_data.data.columns:
            scan_sheet.loc[i, '%s MEAN' %parameter] = np.nanmean(scan_data.data[parameter]) 
                

In [ ]:
scan_sheet

In [ ]:
scan_sheet.to_excel("scanlog.xlsx")